In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModel,
    PreTrainedModel,
    PretrainedConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_full_df['evasion_label'] = train_full_df['evasion_label'].replace('', 'Explicit')
test_df['evasion_label'] = test_df['evasion_label'].replace('', 'Explicit')

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_full_df['evasion_label']
)

In [ ]:
MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 3000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_label2id = {
    "Explicit": 0,
    "Implicit": 1,
    "Dodging": 2,
    "General": 3,
    "Deflection": 4,
    "Partial/half-answer": 5,
    "Declining to answer": 6,
    "Claims ignorance": 7,
    "Clarification": 8
}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}
evasion_labels = list(evasion_label2id.keys())

evasion_to_clarity = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

clarity_to_valid_evasions = {
    "Clear Reply": ["Explicit"],
    "Ambivalent": ["Implicit", "Dodging", "General", "Deflection", "Partial/half-answer"],
    "Clear Non-Reply": ["Declining to answer", "Claims ignorance", "Clarification"],
}

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    
    evasion_ids = []
    for l in examples["evasion_label"]:
        if l == '' or l is None:
            evasion_ids.append(0)  
        else:
            evasion_ids.append(evasion_label2id[l])
    tokenized["evasion_labels"] = evasion_ids
    
    return tokenized

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

In [ ]:
@dataclass
class MultiTaskDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor([f["clarity_labels"] for f in features], dtype=torch.long)
        evasion_labels = torch.tensor([f["evasion_labels"] for f in features], dtype=torch.long)
        
        features_clean = [{k: v for k, v in f.items() if k not in ["clarity_labels", "evasion_labels"]} for f in features]
        
        batch = self.tokenizer.pad(
            features_clean,
            padding=True,
            return_tensors="pt"
        )
        
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = MultiTaskDataCollator(tokenizer=tokenizer)

In [ ]:
class ModernBERTHierarchicalConfig(PretrainedConfig):
    model_type = "modernbert_hierarchical"
    
    def __init__(
        self,
        num_clarity_labels: int = 3,
        num_evasion_labels: int = 9,
        hidden_size: int = 1024,
        hidden_dropout_prob: float = 0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_clarity_labels = num_clarity_labels
        self.num_evasion_labels = num_evasion_labels
        self.hidden_size = hidden_size
        self.hidden_dropout_prob = hidden_dropout_prob


class ModernBERTHierarchical(PreTrainedModel):
    config_class = ModernBERTHierarchicalConfig
    
    def __init__(self, config: ModernBERTHierarchicalConfig):
        super().__init__(config)
        self.config = config
        
        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            reference_compile=False, 
            attn_implementation="sdpa",
            trust_remote_code=True
        )
        
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        self.clarity_classifier = nn.Linear(config.hidden_size, config.num_clarity_labels)

        self.evasion_classifier = nn.Linear(
            config.hidden_size + config.num_clarity_labels, 
            config.num_evasion_labels
        )

        self._init_weights(self.clarity_classifier)
        self._init_weights(self.evasion_classifier)
    
    def _init_weights(self, module):
        """Initialize weights like BERT"""
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if module.bias is not None:
                module.bias.data.zero_()

    def _set_gradient_checkpointing(self, module, value=False):
        """Enable gradient checkpointing for the encoder"""
        if hasattr(self, "encoder"):
            self.encoder.gradient_checkpointing = value
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        return_dict=True,
        **kwargs
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        cls_embedding = outputs.last_hidden_state[:, 0, :]  
        cls_embedding = self.dropout(cls_embedding)

        logits_clarity = self.clarity_classifier(cls_embedding)  
        probs_clarity = torch.softmax(logits_clarity, dim=-1)   

        conditioned_input = torch.cat([cls_embedding, probs_clarity], dim=-1)  
        logits_evasion = self.evasion_classifier(conditioned_input) 
        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss_clarity = loss_fct(logits_clarity, clarity_labels)
            loss_evasion = loss_fct(logits_evasion, evasion_labels)
            loss = loss_clarity + loss_evasion  
        return {
            "loss": loss,
            "logits": logits_evasion,  
            "logits_clarity": logits_clarity,
            "logits_evasion": logits_evasion,
            "probs_clarity": probs_clarity,
        }

In [ ]:

config = ModernBERTHierarchicalConfig(
    num_clarity_labels=3,
    num_evasion_labels=9,
    hidden_size=1024,  
    hidden_dropout_prob=0.1
)

model = ModernBERTHierarchical(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    preds = np.argmax(predictions, axis=-1)
    
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    
    return {'accuracy': accuracy, 'f1': f1}


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_modernbert_hierarchical",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    warmup_ratio=0.05,
    weight_decay=0.10,
    max_grad_norm=1.0,
    gradient_checkpointing=True, 
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch_fused",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    logging_strategy="steps",
    seed=SEED,
    report_to="none",
    label_names=["clarity_labels", "evasion_labels"],
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1", greater_is_better=True)],
)

In [ ]:
trainer.train()

In [ ]:
def tokenize_test(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    
    evasion_ids = []
    for l in examples["evasion_label"]:
        if l == '' or l is None:
            evasion_ids.append(0)
        else:
            evasion_ids.append(evasion_label2id[l])
    tokenized["evasion_labels"] = evasion_ids
    
    return tokenized

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(
    tokenize_test, 
    batched=True, 
    remove_columns=test_dataset.column_names
)

In [ ]:
model.eval()
model.to(device)

all_clarity_preds = []
all_evasion_preds = []
all_clarity_labels = []
all_evasion_labels = []

batch_size = 8
from torch.utils.data import DataLoader

test_dataloader = DataLoader(
    test_tokenized,
    batch_size=batch_size,
    collate_fn=data_collator,
    shuffle=False
)

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        clarity_labels_batch = batch['clarity_labels']
        evasion_labels_batch = batch['evasion_labels']
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        clarity_preds = torch.argmax(outputs['logits_clarity'], dim=-1).cpu().numpy()
        evasion_preds = torch.argmax(outputs['logits_evasion'], dim=-1).cpu().numpy()
        
        all_clarity_preds.extend(clarity_preds)
        all_evasion_preds.extend(evasion_preds)
        all_clarity_labels.extend(clarity_labels_batch.numpy())
        all_evasion_labels.extend(evasion_labels_batch.numpy())

pred_clarity_labels = [clarity_id2label[p] for p in all_clarity_preds]
true_clarity_labels = [clarity_id2label[l] for l in all_clarity_labels]
pred_evasion_labels = [evasion_id2label[p] for p in all_evasion_preds]
true_evasion_labels = [evasion_id2label[l] for l in all_evasion_labels]

In [ ]:
print("Task 1:")

clarity_accuracy = accuracy_score(true_clarity_labels, pred_clarity_labels)
clarity_macro_f1 = f1_score(true_clarity_labels, pred_clarity_labels, average='macro', labels=clarity_labels, zero_division=0)
clarity_weighted_f1 = f1_score(true_clarity_labels, pred_clarity_labels, average='weighted', labels=clarity_labels, zero_division=0)

print(f"Accuracy: {clarity_accuracy:.4f}")
print(f"Macro F1: {clarity_macro_f1:.4f}")
print(f"Weighted F1: {clarity_weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(true_clarity_labels, pred_clarity_labels, labels=clarity_labels, zero_division=0))

print("Confusion Matrix:")
cm_clarity = confusion_matrix(true_clarity_labels, pred_clarity_labels, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm_clarity)

In [ ]:
print("Task 2:")

evasion_accuracy = accuracy_score(true_evasion_labels, pred_evasion_labels)
evasion_macro_f1 = f1_score(true_evasion_labels, pred_evasion_labels, average='macro', labels=evasion_labels, zero_division=0)
evasion_weighted_f1 = f1_score(true_evasion_labels, pred_evasion_labels, average='weighted', labels=evasion_labels, zero_division=0)

print(f"Accuracy: {evasion_accuracy:.4f}")
print(f"Macro F1: {evasion_macro_f1:.4f}")
print(f"Weighted F1: {evasion_weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(true_evasion_labels, pred_evasion_labels, labels=evasion_labels, zero_division=0))

print("Confusion Matrix:")
cm_evasion = confusion_matrix(true_evasion_labels, pred_evasion_labels, labels=evasion_labels)
print(f"Labels: {evasion_labels}")
print(cm_evasion)